# 추출된 획의 구조 오류 진단

**1번 설정 → 2번 준비 → 3번 모듈 로드 → 4번 실행** 순서입니다.
설정은 첫 번째 코드 셀에 모았습니다. 첫 실행은 PNG 1~2개로 확인하세요.

- 기준 획은 누락된 획까지 전부 준비해야 합니다. 입력 획 정답 주석은 사용하지 않습니다.
- 원본 기준과 입력이 정렬돼 있어야 합니다. 그림 첫 줄의 기준 중심선을 확인하세요.
- 정션(J)은 폰트와 무관하게 개별 추출 획의 구조 오류로 봅니다. 짧은 가지도 면제하지 않습니다.
- extraction_error는 D/J 구조 이상 지표이고, reference_deficit은 M/W 기준 대비 부족량입니다.
- 원본 글자 품질은 판정하지 않습니다. 정상 원본에서도 추출기가 정션을 만들 수 있습니다.
- 정션은 분기 반응 근사값이고, 100점 환산이나 사람 선호 교정은 하지 않았습니다.
- 모델 추론은 GPU를 쓸 수 있습니다. 연결 전파는 반복 연산이므로 여러 PNG는 시간이 걸릴 수 있습니다.
- 정상 굵기를 모르는 경우 W는 참고만 하세요. 기본 width_weight=0입니다.
- 이 파일에는 새 평가 코드가 포함돼 있어, GitHub main에 해당 코드가 없어도 실행됩니다.


In [ ]:
# ===== 여기의 설정만 바꾸세요 =====
# "csv": PNG 파일명으로 모델 추론 + 구조 평가
# "saved_npz": 이미 저장된 추론 결과 평가 (모델 추론을 반복하지 않음)
MODE = "csv"
CHECKPOINT = "/content/drive/Shareddrives/2026 자율연구/HSQM/stroke_zip_runs/20261028_082616_066020/best.pt"
CSV_PATH = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/test_dataset/paths.csv"
INPUT_FILES = ["I_아_2.png", "I_야_2.png"]  # 보고 싶은 PNG 파일명. None = 전체 CSV
DATASET_ROOT = None  # CSV 폴더 밖에 이미지가 있으면 데이터의 상위 폴더 지정
NPZ_FILES = []  # saved_npz 모드: ["/content/drive/.../row_0001.npz"]
SAVE_DIR = None  # None = 시간별 새 폴더. 직접 지정할 때는 아직 없는 폴더
SHOW_PLOTS = True

# reference: 원래 기준 좌표. 입력과 기준이 정렬돼 있어야 함.
# hints: 모델이 변형한 기준 좌표. 정렬 오류를 숨길 수 있으므로 비교용으로 사용.
REFERENCE_GEOMETRY = "reference"

# 모든 거리 단위는 추론 해상도의 픽셀 (기본 모델은 128 x 128).
STRUCTURE_CONFIG = dict(
    reference_threshold=0.5,   # 기준 skeleton을 만들 때만 사용하는 이진화 기준
    coverage_radius=2,         # 누락 측정: 기준 경로 주변 위치 오차 허용 반경
    required_ink=0.7,          # 이 값 이상의 잉크 강도를 충분하다고 봄
    anchor_spacing=10,        # 분리 검사 지점 사이의 경로 길이
    anchor_radius=1,          # 시작/도착 검사 영역의 반경
    corridor_radius=3,        # 연결 경로가 이동할 수 있는 기준 주변 반경
    pair_spans=(1, 2),        # 이웃 지점 / 한 지점 건너 연결을 함께 검사
    max_pairs=32,             # 획 하나당 검사할 경로 쌍의 최대 개수
    connectivity_steps=None,  # None = 수렴까지. 작게 제한하면 가짜 끊김 가능
    skeleton_iterations=12,   # soft skeleton 반복 횟수. 두꺼운 획이면 늘릴 것
    junction_radii=(1, 2, 3), # 작은 접합부 분기까지 검사
    junction_margin=0.0,      # 기본값: 폰트/짧은 가지에 대한 정션 면제 없음
    junction_seed_radius=2,   # 기준 주변 본체에서만 연결된 가지를 탐색
    attachment_steps=None,    # None = 수렴까지. 제한 시 수렴 여부가 표에 표시됨
    expected_width=5.0,       # 정상 획의 기대 굵기. 기준 skeleton 굵기가 아님
    width_radius=4,           # 파임 검사 영역 반경
    width_tolerance=0.25,     # 정상 대비 굵기/잉크량 부족 허용 비율
    missing_weight=1.0,       # 별도 reference_deficit 계산에만 사용
    split_weight=1.0,         # 추출 구조 오류의 분리 가중치
    junction_weight=1.0,      # 추출 구조 오류의 정션 가중치
    width_weight=0.0,         # 별도 reference_deficit 계산. 굵기 교정 후 켜기
    eps=1e-6,
)

# 선택적 추출기 추가 학습 API용. 아래 기본 실행은 평가만 하며 학습하지 않습니다.
EXTRACTION_TRAINING_CONFIG = dict(
    structure_weight=0.05,    # 기존 CE/Dice에 추가할 구조 손실 가중치
    mass_guard_weight=0.2,    # 기존 획을 지워서 정션을 피하는 것을 억제
    consistency_weight=0.1,   # 기존 모델의 획 배정을 보존하는 보조 항
    mass_tolerance=0.15,      # 기존 모델 대비 허용하는 획별 잉크 감소 비율
    eps=1e-6,
)


In [ ]:
#@title 2. 기존 V2 모델 코드 준비 (Colab에서 한 번 실행)
from pathlib import Path
import subprocess, sys, tempfile
_structure_repo = Path(tempfile.mkdtemp(prefix="structure_runtime_", dir="/content"))
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/yechan25/hsqm.git", str(_structure_repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(_structure_repo / "requirements.txt")], check=True)
sys.path.insert(0, str(_structure_repo))
for _name in list(sys.modules):
    if _name == "stroke_model" or _name.startswith("stroke_model."):
        del sys.modules[_name]
print("V2 준비 완료. 평가 모듈은 이 노트북에 포함된 버전을 사용합니다.")


In [ ]:
#@title 3. 평가 모듈 로드 (노트북에 코드 포함; GitHub 업로드 불필요)
import sys, types
_structure_sources = {'stroke_model.infer_reference': '"""V2 inference from CSV without input-stroke annotations or label conversion."""\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom stroke_model.convert_reference_csv import Resolver, existing_path, load_ink, path_list, normalized\nfrom stroke_model.reference_model import ReferenceStrokeModel\nfrom stroke_model.preview_reference import show_soft_strokes\n\n\ndef inference_columns(frame):\n    names = {str(c).lower(): c for c in frame.columns}\n    def find(*aliases):\n        for name in aliases:\n            if name.lower() in names:\n                return names[name.lower()]\n        raise ValueError(f\'Missing CSV column: {aliases}; found {list(frame.columns)}\')\n    return {\n        \'image\': find(\'I_prep\', \'input_image_path\', \'input_path\', \'input_image\', \'input\', \'source_path\'),\n        \'reference\': find(\'I_g\', \'reference_image\', \'reference_path\', \'target_image\', \'target_path\', \'glyph_path\'),\n        \'strokes\': find(\'I_G_strokes\', \'reference_strokes\', \'reference_stroke_paths\', \'template_strokes\', \'target_strokes\', \'target_stroke_paths\'),\n    }\n\n\ndef load_inference_row(row, columns, resolve, size):\n    # Deliberately never access input_stroke_paths or total_paths.\n    image_path = resolve(row[columns[\'image\']])\n    reference_path = resolve(row[columns[\'reference\']])\n    paths = path_list(row[columns[\'strokes\']])\n    if not paths:\n        raise ValueError(\'Reference stroke list is empty\')\n    image = torch.from_numpy(load_ink(image_path, size))\n    reference = torch.from_numpy(load_ink(reference_path, size))\n    strokes = torch.from_numpy(np.concatenate([load_ink(resolve(p), size) for p in paths]))\n    if not (strokes.flatten(1).max(1).values > .5).all():\n        raise ValueError(\'Empty reference stroke image\')\n    return image, reference, strokes, image_path, reference_path\n\n\ndef select_input_files(frame, image_column, input_files):\n    if isinstance(input_files, str):\n        input_files = [input_files]\n    values = [normalized(str(value).strip()) for value in frame[image_column]]\n    selected = []\n    for requested in input_files:\n        name = normalized(str(requested).strip())\n        # A bare filename matches a basename; a path matches the CSV path exactly.\n        matches = [i + 1 for i, value in enumerate(values)\n                   if (value == name if \'/\' in name else value.rsplit(\'/\', 1)[-1] == name)]\n        if not matches:\n            raise ValueError(f\'Input filename not found in CSV: {requested}\')\n        if len(matches) != 1:\n            raise ValueError(f\'Ambiguous input filename: {requested}; CSV rows {matches}. Specify the CSV input path or sample_rows.\')\n        if matches[0] not in selected:\n            selected.append(matches[0])\n    return selected\n\n\ndef run_inference_csv(checkpoint, csv_path, sample_rows=None, dataset_root=None,\n                      input_files=None,\n                      save_dir=None, show_soft=True):\n    """sample_rows: one-based CSV data row numbers; None processes all rows.\n\n    Outputs are stroke assignments, not calibrated missing/extra-stroke decisions.\n    No annotation metrics are computed. Existing checkpoints remain unchanged.\n    """\n    import matplotlib.pyplot as plt\n    import json\n\n    checkpoint, csv_path = existing_path(checkpoint), existing_path(csv_path)\n    frame = pd.read_csv(csv_path)\n    columns = inference_columns(frame)\n    if input_files is not None and sample_rows is not None:\n        raise ValueError(\'Specify input_files or sample_rows, not both\')\n    selected = (select_input_files(frame, columns[\'image\'], input_files) if input_files is not None\n                else list(range(1, len(frame) + 1)) if sample_rows is None else list(sample_rows))\n    if not selected or any(not isinstance(n, int) or not 1 <= n <= len(frame) for n in selected):\n        raise ValueError(f\'sample_rows must contain data row numbers in 1..{len(frame)}\')\n    resolve = Resolver(csv_path, existing_path(dataset_root) if dataset_root else csv_path.parent)\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = ReferenceStrokeModel(image_size=state[\'image_size\'], width=state[\'width\'], pretrained=False)\n    model.load_state_dict(state[\'model\'], strict=True)\n    model = model.to(device).eval()\n    size = state[\'image_size\']\n    del state\n    destination = Path(save_dir) if save_dir else checkpoint.parent / (\'inference_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    destination.mkdir(parents=True, exist_ok=False)\n    results = []\n    print(f\'Inference: {len(selected)} samples, {device}. Input stroke annotations are not used.\')\n    for number in selected:\n        try:\n            image, reference, strokes, image_path, reference_path = load_inference_row(\n                frame.iloc[number - 1], columns, resolve, size)\n            with torch.inference_mode():\n                output = model(image[None].to(device), reference[None].to(device), strokes[None].to(device))\n            probabilities = output[\'probabilities\'][0].float().cpu().numpy()\n            masks = output[\'masks\'][0].float().cpu().numpy()\n            prediction = probabilities.argmax(0)\n            ink = image[0].numpy() > .5\n            labels = np.where(ink, prediction, -1)\n            prefix = destination / f\'row_{number:04d}\'\n            np.savez_compressed(str(prefix) + \'.npz\', probabilities=probabilities,\n                                soft_masks=masks, labels=labels, image=image.numpy(),\n                                reference=reference.numpy(), reference_strokes=strokes.numpy())\n            palette = plt.get_cmap(\'tab20\')(np.arange(len(strokes)) % 20)[:, :3]\n            colored = np.ones((*prediction.shape, 3))\n            colored[ink] = palette[prediction[ink]]\n            figure, axes = plt.subplots(1, 3, figsize=(10, 3))\n            for ax, (content, title) in zip(axes, [(image[0], \'Input\'), (reference[0], \'Reference\'), (colored, \'Predicted assignment\')]):\n                ax.imshow(content, cmap=\'gray_r\', vmin=0, vmax=1)\n                ax.set_title(title)\n                ax.axis(\'off\')\n            figure.suptitle(f\'CSV row {number}: {image_path.name} -> {reference_path.name}\')\n            figure.tight_layout()\n            figure.savefig(str(prefix) + \'.png\', dpi=140)\n            plt.show()\n            plt.close(figure)\n            if show_soft:\n                show_soft_strokes(image, strokes, output, prefix=prefix)\n            results.append({\'csv_row\': number, \'input\': str(image_path), \'reference\': str(reference_path),\n                            \'npz\': str(prefix) + \'.npz\', \'preview\': str(prefix) + \'.png\'})\n        except (ValueError, FileNotFoundError) as exc:\n            raise ValueError(f\'CSV data row {number}: {exc}\') from exc\n    report = {\'checkpoint\': str(checkpoint), \'csv\': str(csv_path), \'uses_input_stroke_annotations\': False,\n              \'results\': results}\n    (destination / \'inference.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Saved:\', destination)\n    return report\n', 'stroke_model.stroke_structure': '"""Reference-aware experimental stroke losses; inputs are absolute ink in [0, 1].\n\nReference preparation is discrete and cached. All prediction-dependent losses use\nPyTorch operations, including min/max subgradients. No predicted components are\ndiscarded. Scores are uncalibrated errors, not human-quality probabilities.\n"""\nfrom dataclasses import dataclass, fields\nimport math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, label\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass StructureConfig:\n    # All distances are pixels at the inference resolution (normally 128).\n    reference_threshold: float = 0.5\n    coverage_radius: int = 2\n    required_ink: float = 0.7\n    anchor_spacing: int = 10\n    anchor_radius: int = 1\n    corridor_radius: int = 3\n    pair_spans: tuple = (1, 2)\n    max_pairs: int = 32\n    connectivity_steps: int | None = None  # None: crop area, with convergence check\n    skeleton_iterations: int = 12\n    junction_radii: tuple = (1, 2, 3)\n    junction_margin: float = 0.0  # No font/short-branch exemption by default\n    junction_seed_radius: int = 2\n    attachment_steps: int | None = None  # None: H*W, with convergence check\n    expected_width: float = 5.0  # Optional normal-width prior, NOT skeleton width\n    width_radius: int = 4\n    width_tolerance: float = 0.25\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    width_weight: float = 0.0  # Enable only after normal-width calibration\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name in (\'coverage_radius\', \'anchor_radius\', \'corridor_radius\',\n                     \'junction_seed_radius\', \'width_radius\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        for name in (\'anchor_spacing\', \'max_pairs\', \'skeleton_iterations\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 1:\n                raise ValueError(f\'{name} must be a positive integer\')\n        if self.anchor_spacing <= 2 * self.anchor_radius + 1:\n            raise ValueError(\'anchor_spacing must exceed the anchor diameters\')\n        if not self.pair_spans or any(not isinstance(v, int) or v < 1 for v in self.pair_spans):\n            raise ValueError(\'pair_spans must contain positive integers\')\n        if not self.junction_radii or any(not isinstance(v, int) or v < 1 for v in self.junction_radii):\n            raise ValueError(\'junction_radii must contain integers >= 1\')\n        for name in (\'connectivity_steps\', \'attachment_steps\'):\n            v = getattr(self, name)\n            if v is not None and (not isinstance(v, int) or v < 1):\n                raise ValueError(f\'{name} must be None or a positive integer\')\n        if not 0 < self.required_ink <= 1 or not 0 < self.reference_threshold < 1:\n            raise ValueError(\'required_ink must be in (0,1]; reference_threshold in (0,1)\')\n        if self.expected_width <= 0 or self.eps <= 0 or not 0 <= self.width_tolerance < 1:\n            raise ValueError(\'Invalid width or eps\')\n        if self.junction_margin < 0 or any(getattr(self, n) < 0 for n in\n                (\'missing_weight\', \'split_weight\', \'junction_weight\', \'width_weight\')):\n            raise ValueError(\'Weights and junction_margin must be nonnegative\')\n        if any(not math.isfinite(float(getattr(self, f.name))) for f in fields(self)\n               if isinstance(getattr(self, f.name), (float, int))):\n            raise ValueError(\'Configuration values must be finite\')\n\n\ndef _pool(x, radius, mean=False):\n    k = 2 * radius + 1\n    return (F.avg_pool2d(x, k, 1, radius) if mean else F.max_pool2d(x, k, 1, radius))\n\n\ndef soft_skeleton(x, iterations):\n    """Pooling skeleton from soft-clDice; a surrogate, not exact graph thinning."""\n    def erode(v):\n        return torch.minimum(-F.max_pool2d(-v, (3, 1), 1, (1, 0)),\n                             -F.max_pool2d(-v, (1, 3), 1, (0, 1)))\n    def opening(v):\n        return F.max_pool2d(erode(v), 3, 1, 1)\n    skel = F.relu(x - opening(x))\n    for _ in range(iterations):\n        x = erode(x)\n        delta = F.relu(x - opening(x))\n        skel = skel + F.relu(delta - skel * delta)\n    return skel\n\n\ndef reconstruct(mask, seed, steps=None):\n    """8-connected grayscale reconstruction; no dilation/closing of mask.\n\n    Returns (reachable strength, converged). None allows H*W iterations, a\n    conservative finite-grid bound. Early stopping reads detached values only;\n    the returned tensor retains the graph. Explicit small step caps can truncate.\n    """\n    reached = torch.minimum(mask, seed)\n    limit = mask.shape[-2] * mask.shape[-1] if steps is None else steps\n    converged = False\n    for _ in range(limit):\n        updated = torch.minimum(mask, F.max_pool2d(reached, 3, 1, 1))\n        if torch.equal(updated.detach(), reached.detach()):\n            reached = updated\n            converged = True\n            break\n        reached = updated\n    return reached, converged\n\n\ndef _ordered_centerline(binary):\n    skel = skeletonize(binary)\n    _, count = label(skel, structure=np.ones((3, 3)))\n    if count != 1:\n        raise ValueError(\'Reference stroke must have one connected centerline; clean the reference\')\n    points = [tuple(v) for v in np.argwhere(skel)]\n    lookup = set(points)\n    neighbors = {}\n    for y, x in points:\n        adjacent = []\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                q = (y + dy, x + dx)\n                if (not dy and not dx) or q not in lookup:\n                    continue\n                # Avoid triangular shortcut edges at a normal digital corner.\n                if dy and dx and ((y + dy, x) in lookup or (y, x + dx) in lookup):\n                    continue\n                adjacent.append(q)\n        neighbors[(y, x)] = adjacent\n    if any(len(v) > 2 for v in neighbors.values()):\n        raise ValueError(\'Reference skeleton branches; provide a clean unbranched reference stroke\')\n    endpoints = sorted(p for p, n in neighbors.items() if len(n) == 1)\n    closed = len(points) > 2 and not endpoints\n    if len(points) < 5 or (not closed and len(endpoints) != 2):\n        raise ValueError(\'Reference centerline must be a nontrivial open curve or closed ring\')\n    start = endpoints[0] if endpoints else min(points)\n    order, visited, current = [], set(), start\n    while current not in visited:\n        order.append(current)\n        visited.add(current)\n        choices = sorted(q for q in neighbors[current] if q not in visited)\n        if not choices:\n            break\n        current = choices[0]\n    if len(order) != len(points):\n        raise ValueError(\'Cannot order reference centerline without losing pixels\')\n    return skel, np.asarray(order), closed\n\n\ndef _disk_region(shape, point, radius):\n    yy, xx = np.indices(shape)\n    return ((yy - point[0]) ** 2 + (xx - point[1]) ** 2 <= radius ** 2)\n\n\nclass _Pair(nn.Module):\n    def __init__(self, corridor, a, b, slices):\n        super().__init__()\n        for name, value in [(\'corridor\', corridor), (\'a\', a), (\'b\', b)]:\n            self.register_buffer(name, torch.tensor(value, dtype=torch.float32)[None, None])\n        self.slices = slices\n\n\nclass _Stroke(nn.Module):\n    def __init__(self, reference, config):\n        super().__init__()\n        skel, order, self.closed = _ordered_centerline(reference > config.reference_threshold)\n        self.length = len(order)\n        self.register_buffer(\'skeleton\', torch.tensor(skel, dtype=torch.float32)[None, None])\n        self.register_buffer(\'ys\', torch.tensor(order[:, 0], dtype=torch.long))\n        self.register_buffer(\'xs\', torch.tensor(order[:, 1], dtype=torch.long))\n        distance = distance_transform_edt(~skel)\n        self.register_buffer(\'seeds\', torch.tensor(distance <= config.junction_seed_radius,\n                                                   dtype=torch.float32)[None, None])\n        normal = distance <= max(0.5, config.expected_width / 2)\n        self.register_buffer(\'normal\', torch.tensor(normal, dtype=torch.float32)[None, None])\n        anchors = list(range(0, len(order), config.anchor_spacing))\n        if not self.closed and anchors[-1] != len(order) - 1:\n            anchors.append(len(order) - 1)\n        raw_pairs = []\n        for span in config.pair_spans:\n            for i in range(len(anchors)):\n                j = i + span\n                if not self.closed and j >= len(anchors):\n                    continue\n                if self.closed and span >= len(anchors):\n                    continue\n                begin, end = anchors[i], anchors[j % len(anchors)]\n                path = (order[begin:end + 1] if end > begin else\n                        np.concatenate((order[begin:], order[:end + 1])))\n                if len(path) <= 2 * config.anchor_radius + 2:\n                    continue\n                # Keep anchor patches disjoint, including on tight curves.\n                if np.linalg.norm(order[begin] - order[end]) <= 2 * config.anchor_radius + 1:\n                    continue\n                segment = np.zeros_like(skel)\n                segment[path[:, 0], path[:, 1]] = True\n                corridor = distance_transform_edt(~segment) <= config.corridor_radius\n                a = _disk_region(skel.shape, order[begin], config.anchor_radius) & corridor\n                b = _disk_region(skel.shape, order[end], config.anchor_radius) & corridor\n                yy, xx = np.where(corridor)\n                slices = (slice(yy.min(), yy.max() + 1), slice(xx.min(), xx.max() + 1))\n                raw_pairs.append((corridor[slices], a[slices], b[slices], slices))\n        if len(raw_pairs) > config.max_pairs:\n            ids = np.linspace(0, len(raw_pairs) - 1, config.max_pairs).round().astype(int)\n            raw_pairs = [raw_pairs[i] for i in ids]\n        self.pairs = nn.ModuleList([_Pair(*args) for args in raw_pairs])\n        if not self.pairs:\n            raise ValueError(\'No valid anchor pairs: reduce anchor_spacing/radius for this reference\')\n\n\ndef _ring_offsets(radius):\n    # Sample a digital square perimeter in cyclic order, with no duplicates.\n    return ([( -radius, x) for x in range(-radius, radius)] +\n            [(y, radius) for y in range(-radius, radius)] +\n            [(radius, x) for x in range(radius, -radius, -1)] +\n            [(y, -radius) for y in range(radius, -radius, -1)])\n\n\ndef _crossings(skeleton, radius):\n    h, w = skeleton.shape[-2:]\n    padded = F.pad(skeleton, (radius,) * 4)\n    samples = [padded[..., radius + y:radius + y + h, radius + x:radius + x + w]\n               for y, x in _ring_offsets(radius)]\n    return sum((a - b).abs() for a, b in zip(samples, samples[1:] + samples[:1])) / 2\n\n\nclass StrokeStructureLoss(nn.Module):\n    """Prepare once on fixed aligned reference strokes, then call on live masks.\n\n    reference_strokes: K,H,W. forward: K,H,W or B,K,H,W, floating [0,1].\n    Returned per_stroke values are B,K. loss is an EXTRACTOR structural loss,\n    not a source-image quality score. Reference deficits are returned separately.\n    Closed curves are ordered\n    cyclically; local corridors prevent bypassing a gap via the rest of a ring.\n    No alignment is estimated here. Geometry must already be in input coordinates.\n    """\n    def __init__(self, reference_strokes, config=None):\n        super().__init__()\n        self.config = config or StructureConfig()\n        ref = torch.as_tensor(reference_strokes).detach().float().cpu().numpy()\n        if ref.ndim != 3 or not np.isfinite(ref).all() or (ref < 0).any() or (ref > 1).any():\n            raise ValueError(\'reference_strokes must be finite K,H,W in [0,1]\')\n        self.shape = ref.shape\n        prepared = []\n        for k, stroke in enumerate(ref):\n            try:\n                prepared.append(_Stroke(stroke, self.config))\n            except ValueError as exc:\n                raise ValueError(f\'Reference stroke {k + 1}: {exc}\') from exc\n        if not prepared:\n            raise ValueError(\'At least one reference stroke is required\')\n        self.strokes = nn.ModuleList(prepared)\n\n    def forward(self, masks, return_maps=False):\n        if masks.ndim == 3:\n            masks = masks[None]\n        if masks.ndim != 4 or tuple(masks.shape[1:]) != tuple(self.shape):\n            raise ValueError(f\'masks must have trailing shape {self.shape}\')\n        if not masks.is_floating_point():\n            raise ValueError(\'masks must be floating point\')\n        if not bool(torch.isfinite(masks).all()) or bool((masks < 0).any()) or bool((masks > 1).any()):\n            raise ValueError(\'masks must be finite in [0,1]; do not normalize each stroke\')\n        if next(self.buffers()).device != masks.device:\n            raise ValueError(\'Move the prepared loss to masks.device with .to(device)\')\n        cfg = self.config\n        metrics = {name: [] for name in (\'missing\', \'split\', \'junction\', \'width\')}\n        all_maps, convergence = [], []\n        for k, stroke in enumerate(self.strokes):\n            m = masks[:, k:k + 1].float()\n            coverage = _pool(m, cfg.coverage_radius)[:, 0, stroke.ys, stroke.xs]\n            deficits = F.relu(1 - coverage / cfg.required_ink).square()\n            missing = deficits.mean(-1)\n            local_mass = _pool(m, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            expected = _pool(stroke.normal, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            width = F.relu(1 - local_mass / (expected * (1 - cfg.width_tolerance) + cfg.eps)).square().mean(-1)\n            split_values, split_maps, pair_converged = [], [], []\n            for pair in stroke.pairs:\n                f = m[..., pair.slices[0], pair.slices[1]] * pair.corridor\n                reached, converged = reconstruct(f, f * pair.a, cfg.connectivity_steps)\n                u = (f * pair.a).flatten(1).amax(1)\n                v = (f * pair.b).flatten(1).amax(1)\n                c = (reached * pair.b).flatten(1).amax(1)\n                strength = torch.minimum(u, v)\n                # clamp avoids the tiny nonzero epsilon penalty on perfect paths.\n                gap = F.relu((strength - c) / strength.clamp_min(cfg.eps)).square()\n                split_values.append(u * v * gap)\n                pair_converged.append(converged)\n                if return_maps:\n                    split_maps.append((pair, F.relu(f - reached)))\n            split = torch.stack(split_values, -1).mean(-1)\n            skeleton = soft_skeleton(m, cfg.skeleton_iterations)\n            attached, attach_converged = reconstruct(m, m * stroke.seeds, cfg.attachment_steps)\n            responses = [F.relu(_crossings(skeleton, r) - 2 - cfg.junction_margin).square()\n                         for r in cfg.junction_radii]\n            junction_map = torch.stack(responses).mean(0) * skeleton * attached\n            junction = junction_map.flatten(1).sum(-1) / stroke.length\n            for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction), (\'width\', width)]:\n                metrics[name].append(value)\n            convergence.append({\'stroke\': k + 1, \'pairs\': all(pair_converged),\n                                \'attachment\': attach_converged, \'closed\': stroke.closed})\n            if return_maps:\n                missing_map = torch.zeros_like(m)\n                missing_map[:, 0, stroke.ys, stroke.xs] = deficits\n                split_map = torch.zeros_like(m)\n                for pair, area in split_maps:\n                    # Maps are diagnostics, not the differentiable loss path.\n                    with torch.no_grad():\n                        target = split_map[..., pair.slices[0], pair.slices[1]]\n                        target.copy_(torch.maximum(target, area.detach()))\n                all_maps.append({\'mask\': m.detach(), \'centerline\': stroke.skeleton.detach(),\n                                 \'missing\': missing_map.detach(), \'split\': split_map.detach(),\n                                 \'junction\': junction_map.detach(), \'attached\': attached.detach()})\n        per_stroke = {key: torch.stack(value, -1) for key, value in metrics.items()}\n        extraction = cfg.split_weight * per_stroke[\'split\'] + cfg.junction_weight * per_stroke[\'junction\']\n        reference = cfg.missing_weight * per_stroke[\'missing\'] + cfg.width_weight * per_stroke[\'width\']\n        result = {\'loss\': extraction.mean(), \'reference_loss\': reference.mean(),\n                  \'per_stroke\': {**per_stroke, \'extraction_error\': extraction,\n                                 \'reference_deficit\': reference},\n                  \'source_quality\': {\'assessed\': False,\n                      \'reason\': \'Extracted-mask defects do not identify source-image defects.\'},\n                  \'convergence\': convergence}\n        if return_maps:\n            result[\'maps\'] = all_maps\n        return result\n', 'stroke_model.score_stroke_structure': '"""CSV/NPZ evaluation and plots for the experimental structure losses."""\nfrom dataclasses import asdict\nfrom datetime import datetime\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom stroke_model.stroke_structure import StructureConfig, StrokeStructureLoss\n\n\ndef _destination(save_dir):\n    destination = Path(save_dir) if save_dir else Path(\'structure_results\') / datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')\n    destination.mkdir(parents=True, exist_ok=False)\n    return destination\n\n\ndef _measure(masks, geometry, config, prefix, reference=None, image=None,\n             show=True, caption=\'\'):\n    evaluator = StrokeStructureLoss(geometry, config).to(masks.device)\n    with torch.no_grad():\n        result = evaluator(masks, return_maps=True)\n    rows = []\n    for k in range(len(geometry)):\n        rows.append({\'stroke\': k + 1,\n                     **{key: float(value[0, k]) for key, value in result[\'per_stroke\'].items()},\n                     \'closed_reference\': result[\'convergence\'][k][\'closed\'],\n                     \'connectivity_converged\': result[\'convergence\'][k][\'pairs\'],\n                     \'attachment_converged\': result[\'convergence\'][k][\'attachment\']})\n    frame = pd.DataFrame(rows)\n    frame.to_csv(str(prefix) + \'_errors.csv\', index=False, encoding=\'utf-8-sig\')\n    payload = {\'soft_masks\': masks.detach().float().cpu().numpy(),\n               \'reference_geometry\': torch.as_tensor(geometry).detach().cpu().numpy()}\n    if reference is not None:\n        payload[\'reference_strokes\'] = torch.as_tensor(reference).detach().cpu().numpy()\n    if image is not None:\n        payload[\'image\'] = torch.as_tensor(image).detach().cpu().numpy()\n    for name in (\'centerline\', \'missing\', \'split\', \'junction\', \'attached\'):\n        payload[name + \'_maps\'] = np.stack([m[name][0, 0].cpu().numpy() for m in result[\'maps\']])\n    np.savez_compressed(str(prefix) + \'_structure.npz\', **payload)\n    figures = _plot(result[\'maps\'], rows, prefix, caption, show)\n    print(caption)\n    print(frame.round(5).to_string(index=False))\n    if not frame[\'connectivity_converged\'].all() or not frame[\'attachment_converged\'].all():\n        print(\'Iteration cap reached: increase connectivity_steps/attachment_steps before interpreting these errors.\')\n    return {\'strokes\': rows, \'mean_errors\': {key: float(value.mean()) for key, value in result[\'per_stroke\'].items()},\n            \'worst_extraction_error\': float(result[\'per_stroke\'][\'extraction_error\'].max()),\n            \'source_quality\': result[\'source_quality\'],\n            \'csv\': str(prefix) + \'_errors.csv\', \'npz\': str(prefix) + \'_structure.npz\', \'figures\': figures}\n\n\ndef _plot(maps, rows, prefix, caption, show):\n    import matplotlib.pyplot as plt\n    paths = []\n    for offset in range(0, len(maps), 4):\n        page = maps[offset:offset + 4]\n        fig, axes = plt.subplots(5, len(page), squeeze=False, figsize=(3.2*len(page), 12))\n        for col, item in enumerate(page):\n            k = offset + col\n            for row, (name, title) in enumerate([\n                    (\'centerline\', \'Reference centerline\'), (\'mask\', \'Soft mask (absolute ink)\'),\n                    (\'missing\', \'Missing on reference path\'),\n                    (\'split\', \'Extractor connectivity diagnostic\'), (\'junction\', \'Extractor junction defect\')]):\n                content = item[name][0, 0].cpu().numpy()\n                ax = axes[row, col]\n                if row >= 2:\n                    ax.imshow(item[\'mask\'][0, 0].cpu().numpy(), cmap=\'gray_r\', vmin=0, vmax=1, alpha=.25)\n                    artist = ax.imshow(np.ma.masked_less_equal(content, 1e-8), cmap=\'magma\',\n                                       vmin=0, vmax=1 if name != \'junction\' else max(1., float(content.max())))\n                    fig.colorbar(artist, ax=ax, fraction=.04, pad=.02)\n                else:\n                    ax.imshow(content, cmap=\'gray_r\', vmin=0, vmax=1)\n                ax.set_title(f\'Stroke {k+1}: {title}\', fontsize=9)\n                ax.axis(\'off\')\n            values = rows[k]\n            axes[4, col].text(.5, -.08,\n                f"M={values[\'missing\']:.4f}  D={values[\'split\']:.4f}\\n"\n                f"J={values[\'junction\']:.4f}  W={values[\'width\']:.4f}",\n                transform=axes[4, col].transAxes, ha=\'center\', va=\'top\', fontsize=9)\n        fig.suptitle(caption + \'\\nExtractor defects + reference deficits. Source-image quality is NOT assessed.\', fontsize=10)\n        fig.tight_layout(rect=(0, .025, 1, .96))\n        path = str(prefix) + f\'_structure_{offset//4+1}.png\'\n        fig.savefig(path, dpi=140)\n        paths.append(path)\n        if show:\n            plt.show()\n        plt.close(fig)\n    return paths\n\n\ndef score_saved_strokes(npz_paths, config=None, save_dir=None, show=True):\n    """Use saved run_inference_csv NPZ files: soft_masks + reference_strokes.\n\n    Also accepts the inference report dictionary returned by run_inference_csv.\n    Uses unwarped reference geometry: input and reference must already be aligned.\n    """\n    config = config or StructureConfig()\n    if isinstance(npz_paths, dict):\n        npz_paths = [item[\'npz\'] for item in npz_paths[\'results\']]\n    elif isinstance(npz_paths, (str, Path)):\n        npz_paths = [npz_paths]\n    npz_paths = list(npz_paths)\n    if not npz_paths:\n        raise ValueError(\'No NPZ files selected\')\n    destination = _destination(save_dir)\n    results = []\n    for i, path in enumerate(npz_paths):\n        with np.load(path, allow_pickle=False) as data:\n            masks = torch.tensor(data[\'soft_masks\'], dtype=torch.float32)\n            reference = torch.tensor(data[\'reference_strokes\'], dtype=torch.float32)\n            image = data[\'image\'].copy() if \'image\' in data else None\n        if masks.ndim == 4 and masks.shape[0] == 1:\n            masks = masks[0]\n        if masks.ndim != 3:\n            raise ValueError(f\'Expected K,H,W masks in {path}\')\n        results.append({\'input_npz\': str(path), **_measure(masks, reference, config,\n            destination / f\'sample_{i+1:04d}\', reference=reference, image=image,\n            show=show, caption=Path(path).name)})\n    report = {\'mode\': \'saved_npz\', \'geometry\': \'reference\', \'config\': asdict(config),\n              \'source_quality\': {\'assessed\': False}, \'results\': results}\n    (destination / \'structure_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structure results:\', destination)\n    return report\n\n\ndef score_structure_csv(checkpoint, csv_path, input_files=None, sample_rows=None,\n                        dataset_root=None, config=None, geometry=\'reference\',\n                        save_dir=None, show=True):\n    """Predict selected PNGs then measure errors, ignoring target annotations.\n\n    geometry=\'reference\': fixed reference coordinates, requires alignment.\n    geometry=\'hints\': detached model-warped reference; can hide alignment errors.\n    Evaluation only. For guidance call StrokeStructureLoss on non-detached masks.\n    """\n    from stroke_model.convert_reference_csv import Resolver, existing_path\n    from stroke_model.infer_reference import inference_columns, load_inference_row, select_input_files\n    from stroke_model.reference_model import ReferenceStrokeModel\n\n    if geometry not in (\'reference\', \'hints\'):\n        raise ValueError(\'geometry must be reference or hints\')\n    if input_files is not None and sample_rows is not None:\n        raise ValueError(\'Specify input_files or sample_rows, not both\')\n    config = config or StructureConfig()\n    checkpoint, csv_path = existing_path(checkpoint), existing_path(csv_path)\n    frame = pd.read_csv(csv_path)\n    columns = inference_columns(frame)\n    selected = (select_input_files(frame, columns[\'image\'], input_files) if input_files is not None else\n                list(range(1, len(frame)+1)) if sample_rows is None else list(sample_rows))\n    if not selected or any(not isinstance(i, int) or not 1 <= i <= len(frame) for i in selected):\n        raise ValueError(f\'Select CSV rows in 1..{len(frame)}\')\n    resolve = Resolver(csv_path, existing_path(dataset_root) if dataset_root else csv_path.parent)\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    model = ReferenceStrokeModel(image_size=state[\'image_size\'], width=state[\'width\'], pretrained=False)\n    model.load_state_dict(state[\'model\'], strict=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = model.to(device).eval()\n    size = state[\'image_size\']\n    del state\n    destination = _destination(save_dir or checkpoint.parent / (\'structure_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')))\n    print(\'Checkpoint:\', checkpoint)\n    print(\'CSV:\', csv_path)\n    print(\'Reference geometry:\', geometry, \'/ annotations are not used\')\n    results = []\n    for number in selected:\n        image, reference, strokes, image_path, _ = load_inference_row(frame.iloc[number-1], columns, resolve, size)\n        with torch.no_grad():\n            output = model(image[None].to(device), reference[None].to(device), strokes[None].to(device))\n        # Reference preparation uses CPU only; scalar scoring runs on model device.\n        prepared_geometry = strokes if geometry == \'reference\' else output[\'hints\'][0].detach().cpu()\n        result = _measure(output[\'masks\'][0], prepared_geometry, config,\n            destination / f\'row_{number:04d}\', reference=strokes, image=image,\n            show=show, caption=f\'CSV row {number}: {image_path.name}\')\n        results.append({\'csv_row\': number, \'input\': str(image_path), **result})\n    report = {\'mode\': \'csv\', \'checkpoint\': str(checkpoint), \'csv\': str(csv_path),\n              \'geometry\': geometry, \'config\': asdict(config), \'uses_input_stroke_annotations\': False,\n              \'source_quality\': {\'assessed\': False}, \'results\': results}\n    (destination / \'structure_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structure results:\', destination)\n    return report\n', 'stroke_model.extraction_training': '"""Supervised extractor fine-tuning objective. Never a source-quality reward.\n\nThe caller supplies a live StrokeStructureLoss result using fixed, aligned\nreference geometry and a frozen teacher prediction for the same batch.\n"""\nfrom dataclasses import dataclass\nimport math\n\nimport torch\nfrom .reference_model import partition_loss\n\n\n@dataclass(frozen=True)\nclass ExtractionTrainingConfig:\n    structure_weight: float = 0.05\n    mass_guard_weight: float = 0.2\n    consistency_weight: float = 0.1\n    mass_tolerance: float = 0.15\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name in (\'structure_weight\', \'mass_guard_weight\', \'consistency_weight\'):\n            value = getattr(self, name)\n            if not math.isfinite(value) or value < 0:\n                raise ValueError(f\'{name} must be finite and nonnegative\')\n        if not 0 <= self.mass_tolerance < 1 or not math.isfinite(self.eps) or self.eps <= 0:\n            raise ValueError(\'Invalid mass_tolerance or eps\')\n\n\ndef extraction_training_loss(output, labels, structure_result=None,\n                             teacher_output=None, config=None, *, topology_valid=None):\n    """Base CE/Dice + extractor topology + teacher safeguards.\n\n    Missing/width reference deficits are deliberately excluded. A real missing\n    input stroke must not be fabricated by this extractor (masks=image*p).\n    topology_valid is explicit B,K metadata: only known structurally complete\n    source strokes may receive the no-junction/connectedness regularizer. Do not\n    derive this metadata from the student\'s own predictions.\n    Teacher safeguards reduce deletion/channel-transfer shortcuts but do not\n    prove their absence. Validate supervised Dice AND structural diagnostics.\n    """\n    cfg = config or ExtractionTrainingConfig()\n    base = partition_loss(output, labels)\n    zero = output[\'masks\'].sum() * 0\n    if cfg.structure_weight == 0:\n        return {**base, \'supervised\': base[\'loss\'], \'structure\': zero,\n                \'mass_guard\': zero, \'consistency\': zero}\n    if structure_result is None or teacher_output is None:\n        raise ValueError(\'Structural fine-tuning requires live structure_result and frozen teacher_output\')\n    if cfg.mass_guard_weight <= 0:\n        raise ValueError(\'Enable mass_guard_weight to protect against deleting strokes\')\n    if not all(c[\'pairs\'] and c[\'attachment\'] for c in structure_result[\'convergence\']):\n        raise ValueError(\'Structure propagation did not converge; increase iteration caps\')\n    masks = output[\'masks\'].float()\n    teacher = teacher_output[\'masks\'].detach().to(device=masks.device, dtype=torch.float32)\n    active = output[\'active\']\n    if teacher.shape != masks.shape or active.shape != masks.shape[:2]:\n        raise ValueError(\'Student/teacher mask shapes and active channels must match\')\n    if topology_valid is None or topology_valid.shape != active.shape or topology_valid.dtype != torch.bool:\n        raise ValueError(\'Provide topology_valid: B,K bool metadata for known-complete input strokes\')\n    valid = active & topology_valid.to(active.device)\n    if not bool(torch.isfinite(teacher).all()) or bool((teacher < 0).any()) or bool((teacher > 1).any()):\n        raise ValueError(\'Teacher masks must be finite absolute ink in [0,1]\')\n    errors = structure_result[\'per_stroke\'][\'extraction_error\']\n    if errors.shape != active.shape:\n        raise ValueError(\'Structure results must be B,K for the same live output\')\n    if masks.requires_grad and not errors.requires_grad:\n        raise ValueError(\'Do not detach structural losses in extractor training\')\n    structure = errors[valid].mean() if bool(valid.any()) else zero\n    target_mass = teacher.sum((-2, -1))\n    mass = masks.sum((-2, -1))\n    guarded = valid & (target_mass > cfg.eps)\n    deficit = torch.relu(((1 - cfg.mass_tolerance) * target_mass - mass)\n                         / target_mass.clamp_min(cfg.eps)).square()\n    mass_guard = deficit[guarded].mean() if bool(guarded.any()) else zero\n    # Only reference channel activity is discrete. Teacher tensors are detached.\n    consistency = (masks - teacher).square().sum((-2, -1))\n    consistency = (consistency / target_mass.clamp_min(1))[valid].mean() if bool(valid.any()) else zero\n    loss = (base[\'loss\'] + cfg.structure_weight * structure\n            + cfg.mass_guard_weight * mass_guard + cfg.consistency_weight * consistency)\n    return {**base, \'loss\': loss, \'supervised\': base[\'loss\'], \'structure\': structure,\n            \'mass_guard\': mass_guard, \'consistency\': consistency}\n'}
for _module_name, _source in _structure_sources.items():
    _module = types.ModuleType(_module_name)
    _module.__package__ = "stroke_model"
    _module.__file__ = "<embedded:" + _module_name + ">"
    sys.modules[_module_name] = _module
    exec(compile(_source, _module.__file__, "exec"), _module.__dict__)
print("구조 평가 모듈 로드 완료")


In [ ]:
#@title 4. 실행: PNG 선택 → 획별 오류 표와 위치 그림
from stroke_model.stroke_structure import StructureConfig
from stroke_model.score_stroke_structure import score_structure_csv, score_saved_strokes

structure_config = StructureConfig(**STRUCTURE_CONFIG)
if MODE == "csv":
    from google.colab import drive
    drive.mount("/content/drive")
    structure_results = score_structure_csv(
        CHECKPOINT, CSV_PATH, input_files=INPUT_FILES, dataset_root=DATASET_ROOT,
        config=structure_config, geometry=REFERENCE_GEOMETRY,
        save_dir=SAVE_DIR, show=SHOW_PLOTS,
    )
elif MODE == "saved_npz":
    selected_npz = NPZ_FILES or globals().get("inference_results")
    if not selected_npz:
        raise ValueError("NPZ_FILES에 기존 추론 NPZ 경로를 넣으세요.")
    # 기존 셀에서 Drive를 연결하지 않았다면 먼저 연결하세요.
    structure_results = score_saved_strokes(
        selected_npz, config=structure_config, save_dir=SAVE_DIR, show=SHOW_PLOTS,
    )
else:
    raise ValueError('MODE는 "csv" 또는 "saved_npz"여야 합니다.')


## 선택적 추출기 추가 학습 API

정션은 우선 추출기 개선용 신호입니다. 원본 글자 품질 보상으로 역확산에 직접 연결하지 않습니다.
다음은 학습 루프에 넣을 손실 구성 예시이며 위 평가 셀은 가중치를 갱신하지 않습니다.
배치의 기준 geometry는 입력 좌표에 고정 정렬되어 있어야 합니다. 샘플마다 다르면 criterion도 샘플별로 준비해야 합니다.

```python
from copy import deepcopy
from stroke_model.stroke_structure import StrokeStructureLoss
from stroke_model.extraction_training import ExtractionTrainingConfig, extraction_training_loss

# 추가 학습 전에 한 번만 생성합니다.
teacher = deepcopy(extractor).eval().requires_grad_(False)
criterion = StrokeStructureLoss(aligned_reference_strokes, structure_config).to(device)
train_config = ExtractionTrainingConfig(**EXTRACTION_TRAINING_CONFIG)

# 정답 labels가 있는 학습 배치에서 사용합니다.
extractor.train()
with torch.no_grad():
    teacher_output = teacher(image, reference, reference_strokes)
output = extractor(image, reference, reference_strokes)
structure = criterion(output["masks"])  # student 경로는 detach/no_grad 금지
# B,K bool 주석: 원본에서 정상적으로 완전한 획임을 확인한 채널만 True.
# 실제 누락/손상 또는 정상 여부가 불명확한 원본 채널은 False로 지정합니다.
losses = extraction_training_loss(output, labels, structure, teacher_output, train_config,
                                 topology_valid=known_complete_strokes)
optimizer.zero_grad(set_to_none=True)
losses["loss"].backward()
optimizer.step()
```

기존 정답 CE/Dice + 구조 손실 + 잉크 감소 방지 + 기존 출력 보존을 함께 사용합니다.
M/W는 이 학습 손실에 넣지 않습니다. 입력에서 실제로 없어진 획을 만들어내게 해서는 안 됩니다.
교차점의 주석 공백과 기존 모델 오류가 남으므로, 정답 Dice와 구조 오류를 함께 검증해야 합니다.
